# Pointless cubic surfaces: watching the Riemann–Roch recipe fail

Companion to `riemann-roch-spaces.ipynb`, whose §II produced a birational map $S \dashrightarrow \mathbb{P}^2$
from a Galois-stable set of three skew lines, and whose §II.7 argued that the Severi–Brauer obstruction
is the arithmetic content rather than a defect of the method. Here we take surfaces with **no rational
points** and watch the recipe fail, in the two cases:

* **(a)** a $G$-stable set of **three** skew lines, but **no** stable sixer;
* **(b)** a $G$-stable **sixer**, but no stable set of three skew lines.

The two fail in genuinely different places. In (a) there is no invariant divisor class to feed the
recipe at all — the failure is in $\operatorname{Pic}(\bar S)^{G}$, before $\operatorname{Br}$ is reached. In (b) the
class exists and the obstruction is exactly the Brauer class of the Severi–Brauer surface.

## 1. Both cases occur: the combinatorics in $W(E_6)$

First, purely combinatorially. The $27$ lines carry the Schläfli graph, whose automorphism group is
$W(E_6)$; a sixer has stabiliser $\cong S_6$, and $72 \cdot 720 = 51840$.

In [1]:
from itertools import combinations
Mq = diagonal_matrix(ZZ, [1,-1,-1,-1,-1,-1,-1])
def dt(a,b): return vector(ZZ,a)*Mq*vector(ZZ,b)
lc = (1,0,0,0,0,0,0)
Ee = [tuple([0]+[1 if j==i else 0 for j in range(6)]) for i in range(6)]
Kc = tuple(vector(ZZ,(-3,0,0,0,0,0,0)) + sum(vector(ZZ,e) for e in Ee))
AL = list(Ee) \
   + [tuple(vector(ZZ,lc)-vector(ZZ,Ee[i])-vector(ZZ,Ee[j])) for i in range(6) for j in range(i+1,6)] \
   + [tuple(2*vector(ZZ,lc)-sum(vector(ZZ,Ee[j]) for j in range(6) if j!=i)) for i in range(6)]
skA = [[dt(AL[i],AL[j])==0 for j in range(27)] for i in range(27)]

def cliq(sk, size):
    out=[]
    def rec(cur, st):
        if len(cur)==size: out.append(tuple(cur)); return
        for v in range(st,27):
            if all(sk[v][u] for u in cur): rec(cur+[v], v+1)
    rec([],0); return out

triA, sixA = cliq(skA,3), cliq(skA,6)
print("triples of pairwise skew lines:", len(triA), "   sixers:", len(sixA))
GrA = Graph([(i,j) for i in range(27) for j in range(i+1,27) if not skA[i][j]])
W = GrA.automorphism_group()
print("Aut(Schlafli graph) = W(E_6), order", W.order(), "  = 72 * 720 =", 72*720)

triples of pairwise skew lines: 720    sixers: 72
Aut(Schlafli graph) = W(E_6), order 51840   = 72 * 720 = 51840


In [2]:
def induced(sig):
    "the permutation of the 27 lines induced by sig in S_6 acting on E_1..E_6"
    M = Matrix(ZZ, [list(lc)] + [list(Ee[sig[i]]) for i in range(6)]).transpose()
    pos = {AL[i]: i for i in range(27)}
    return tuple(pos[tuple(M*vector(ZZ,AL[i]))] for i in range(27))

S6 = SymmetricGroup(6)
print("subgroups G of the sixer stabiliser S_6 with NO stable triple  (= case (b)):")
print("   group            order   stable triples   stable sixers")
caseb = 0
for H in S6.conjugacy_classes_subgroups():
    gens = [induced(tuple(g(i+1)-1 for i in range(6))) for g in H.gens()] or [tuple(range(27))]
    st = lambda c: all(set(p[i] for i in c) == set(c) for p in gens)
    t = sum(1 for c in triA if st(c)); s = sum(1 for c in sixA if st(c))
    if t == 0:
        caseb += 1
        if H.order() >= 36:
            print("   " + str(H.structure_description()).ljust(16) + str(H.order()).rjust(5)
                  + str(t).rjust(15) + str(s).rjust(15))
print("\n" + str(caseb), "conjugacy classes of subgroups realise case (b);")
print("the full stabiliser S_6 is one of them, so case (b) is possible.")

subgroups G of the sixer stabiliser S_6 with NO stable triple  (= case (b)):
   group            order   stable triples   stable sixers


   S3 x S3            36              0              2
   (C3 x C3) : C4     36              0              2
   C2 x S4            48              0              2
   C2 x S4            48              0              2
   A5                 60              0              2
   A5                 60              0              2
   (S3 x S3) : C2     72              0              2
   S5                120              0              2
   S5                120              0              2
   A6                360              0              2
   S6                720              0              2

37 conjugacy classes of subgroups realise case (b);
the full stabiliser S_6 is one of them, so case (b) is possible.


So a sixer can be stable with no stable triple anywhere — the Galois group has only to act on the six
lines without an invariant $3$-subset (and $S_6$, $A_6$, $S_5$, $A_5$, $S_3\times S_3$, $S_4$, … all
do). Case (a) is realised arithmetically in §2 below.

## 2. Case (a): three skew lines, no sixer, no rational points

Take the diagonal surface
$$S : \; x^3 + 2y^3 + 7z^3 + 14w^3 = 0 .$$
Its $27$ lines are completely explicit. For each of the three ways of splitting the coordinates into
two pairs, the lines are $x_i = \theta x_j$, $x_k = \mu x_l$ with $\theta^3 = -a_j/a_i$ and
$\mu^3 = -a_l/a_k$; and two such are skew exactly when they differ in *both* cube roots (within a
pair-splitting), with explicit conditions across splittings. Since every cube root here is a
monomial in $2^{1/3}, 7^{1/3}$ times a cube root of unity, the whole Galois action can be recorded
exactly, with no number-field arithmetic: an element of
$\operatorname{Gal}(\mathbb{Q}(\zeta_3,2^{1/3},7^{1/3})/\mathbb{Q})$ is
$2^{1/3}\mapsto\zeta^{a}2^{1/3}$, $7^{1/3}\mapsto\zeta^{b}7^{1/3}$, $\zeta\mapsto\zeta^{\varepsilon}$.

In [3]:
from itertools import product
PRIMES = [2,7]
def fact(q):
    q=QQ(q); s = 1 if q>0 else -1; q=abs(q)
    e=[q.valuation(p) for p in PRIMES]
    assert q == prod(p^ee for p,ee in zip(PRIMES,e))
    return (s, tuple(e))
def croot(q):
    "principal cube root of a rational, as (sign, exponents-over-3, power of zeta)"
    s,e = fact(q); return (s, tuple(e), 0)
def mul(A,B):  return (A[0]*B[0], tuple(u+v for u,v in zip(A[1],B[1])), (A[2]+B[2])%3)
def zmul(A,k): return (A[0], A[1], (A[2]+k)%3)
def sig(A,co,eps): return (A[0], A[1], (eps*A[2] + sum(c*u for c,u in zip(co,A[1])))%3)
PART = [((0,1),(2,3)), ((0,2),(1,3)), ((0,3),(1,2))]

def lines_and_galois(a, local=False):
    L=[]
    for p,((i,j),(k,l)) in enumerate(PART):
        th0 = croot(-QQ(a[j])/a[i]); mu0 = croot(-QQ(a[l])/a[k])
        for kt in range(3):
            for km in range(3): L.append((p,kt,km,zmul(th0,kt),zmul(mu0,km)))
    def meet(X,Y):
        p,kt,km,th,mu = X; q,lt,lm,th2,mu2 = Y
        if p==q: return kt==lt or km==lm
        if (p,q)==(0,1): return mul(th,mu2)==mul(th2,mu)
        if (p,q)==(1,0): return mul(th2,mu)==mul(th,mu2)
        if (p,q)==(0,2): return th2==mul(mul(th,mu),mu2)
        if (p,q)==(2,0): return th==mul(mul(th2,mu2),mu)
        if (p,q)==(1,2): return mul(th2,mu2)==mul(th,mu)
        return mul(th,mu)==mul(th2,mu2)
    idx = {(l[0],l[1],l[2]): i for i,l in enumerate(L)}
    eps = [1] if local else [1,2]
    perms = sorted(set(tuple(idx[(l[0], sig(l[3],co,e)[2], sig(l[4],co,e)[2])] for l in L)
                       for co in product(range(3),repeat=len(PRIMES)) for e in eps))
    sk = [[not meet(L[i],L[j]) for j in range(27)] for i in range(27)]
    return L, perms, sk

a = (1,2,7,14)
L, perms, sk = lines_and_galois(a)
print("every line meets exactly 10 others (Schlafli):",
      set(sum(1 for j in range(27) if not sk[i][j] and j!=i) for i in range(27)))
print("order of the Galois image on the 27 lines:", len(perms))
tri, six = cliq(sk,3), cliq(sk,6)
stab = lambda c: all(set(p[i] for i in c)==set(c) for p in perms)
st3 = [c for c in tri if stab(c)]; st6 = [c for c in six if stab(c)]
print("stable triples of skew lines:", len(st3), "   stable sixers:", len(st6))

every line meets exactly 10 others (Schlafli): {10}
order of the Galois image on the 27 lines: 18
stable triples of skew lines: 2    stable sixers: 0


In [4]:
def pointless_at(a,p):
    "diagonal cubic in 4 variables over Q_p, p = 1 mod 3: is it insoluble ?"
    if p % 3 != 1: return False
    cls = {}
    for i,c in enumerate(a): cls.setdefault(ZZ(c).valuation(p) % 3, []).append(i)
    for k, ix in cls.items():
        if len(ix) >= 3: return False                       # a smooth plane cubic over F_p has points
        if len(ix) == 2:
            i,j = ix; r = -QQ(a[j])/a[i]; r = r/p^(QQ(r).valuation(p))
            if GF(p)(r)^((p-1)//3) == 1: return False       # Hensel
    return True

print("v_7 of the coefficients:", [ZZ(c).valuation(7) for c in a], " -> classes mod 3: 0,0,1,1")
print("the two ratios that could cancel:  -a1/a0 =", -QQ(a[1])/a[0],
      "   -a3/a2 =", -QQ(a[3])/a[2])
print("is -2 a cube mod 7 ?", GF(7)(-2)^2 == 1, "   (cubes mod 7 are", sorted(set(GF(7)(u)^3 for u in range(1,7))), ")")
print()
print("S(Q_7) empty ?", pointless_at(a,7), "   =>  S(Q) is empty")

v_7 of the coefficients: [0, 0, 1, 1]  -> classes mod 3: 0,0,1,1
the two ratios that could cancel:  -a1/a0 = -2    -a3/a2 = -2
is -2 a cube mod 7 ? False    (cubes mod 7 are [1, 6] )

S(Q_7) empty ? True    =>  S(Q) is empty


The two stable triples are visible by hand. For the splitting $\{0,1\},\{2,3\}$ we get
$\theta^3 = -2$ and $\mu^3 = -14/7 = -2$: the *same* cubic algebra, so matching $\mu = \theta$ is
Galois-equivariant and
$$T_2 = \big\{\, x_0 = \theta x_1,\; x_2 = \theta x_3 \;:\; \theta^3 = -2 \,\big\}$$
is a Galois orbit of three pairwise skew lines. For $\{0,2\},\{1,3\}$, $\theta^3 = -7$ and
$\mu^3 = -14/2 = -7$, giving
$$T_7 = \big\{\, x_0 = \theta x_2,\; x_1 = \theta x_3 \;:\; \theta^3 = -7 \,\big\} .$$
For the third splitting the two algebras are $\mathbb{Q}(14^{1/3})$ and $\mathbb{Q}(28^{1/3})$, which
differ, so there is no third triple — matching the count of $2$ above.

In [5]:
for (p,name,c1,c2) in [(0,"T_2",-QQ(a[1])/a[0],-QQ(a[3])/a[2]),
                       (1,"T_7",-QQ(a[2])/a[0],-QQ(a[3])/a[1]),
                       (2,"--- ",-QQ(a[3])/a[0],-QQ(a[2])/a[1])]:
    print(name, ": theta^3 =", str(c1).rjust(6), "  mu^3 =", str(c2).rjust(6),
          "   same cubic algebra ?", c1 == c2)
print()
print("the two stable triples, as index sets of lines:", st3)
print("each is a single Galois orbit ?",
      [len(set(tuple(sorted(p[i] for i in c)) for p in perms)) == 1 for c in st3])

T_2 : theta^3 =     -2   mu^3 =     -2    same cubic algebra ? True
T_7 : theta^3 =     -7   mu^3 =     -7    same cubic algebra ? True
---  : theta^3 =    -14   mu^3 =   -7/2    same cubic algebra ? False

the two stable triples, as index sets of lines: [(0, 4, 8), (9, 13, 17)]
each is a single Galois orbit ? [True, True]


### Where the recipe dies

Now transport the Galois action to $\operatorname{Pic}(\bar S) \cong \mathbb{Z}^7$ (match the concrete Schläfli
graph with the abstract one, read off the images of a sixer, and use
$\ell = (-K + \sum E_i)/3$). The recipe of the companion worksheet needs an **invariant class $D$ with
$D^2 = 1$, $D\cdot K = -3$**; there is none.

In [6]:
GrC = Graph([(i,j) for i in range(27) for j in range(i+1,27) if not sk[i][j]])
ok, cert = GrC.is_isomorphic(GrA, certificate=True)
print("concrete lines carry the Schlafli graph:", ok)
mats = []
for p in perms:
    q = [0]*27
    for i in range(27): q[cert[i]] = cert[p[i]]
    imE = [AL[q[i]] for i in range(6)]
    iml = (-vector(ZZ,Kc) + sum(vector(ZZ,e) for e in imE))/3
    mats.append(Matrix(ZZ, [list(iml)] + [list(e) for e in imE]).transpose())
print("the matrices preserve the intersection form and fix K:",
      all(M.transpose()*Mq*M == Mq and M*vector(ZZ,Kc) == vector(ZZ,Kc) for M in mats))
Fix = Matrix(ZZ,0,7)
for M in mats: Fix = Fix.stack(M - identity_matrix(7))
inv = Fix.right_kernel()
print("rank of Pic(Sbar)^G :", inv.dimension(), "   basis:", [tuple(v) for v in inv.basis()])

concrete lines carry the Schlafli graph: True
the matrices preserve the intersection form and fix K: True
rank of Pic(Sbar)^G : 2    basis: [(1, 3, -3, -1, -1, 1, 1), (0, 5, -4, -1, -1, 2, 2)]


In [7]:
def bdclass(s):
    B = Matrix(ZZ, [[dt(tuple(b), AL[i]) for b in identity_matrix(ZZ,7).rows()] for i in s]
                 + [[dt(tuple(b), Kc)    for b in identity_matrix(ZZ,7).rows()]])
    return tuple(B.solve_right(vector(ZZ,[0]*6 + [-3])))
BD = [bdclass(s) for s in sixA]
invBD = [D for D in BD if all(M*vector(ZZ,D) == vector(ZZ,D) for M in mats)]
print("blow-down classes (D^2 = 1, D.K = -3):", len(BD))
print("of which Galois-invariant:", len(invBD))
print()
print("=> there is no class l to start from: the recipe fails in Pic(Sbar)^G,")
print("   before any Brauer class is even defined.  S is birational to a MINIMAL")
print("   del Pezzo surface of degree 6, not to a Severi-Brauer surface.")

blow-down classes (D^2 = 1, D.K = -3): 72
of which Galois-invariant: 0

=> there is no class l to start from: the recipe fails in Pic(Sbar)^G,
   before any Brauer class is even defined.  S is birational to a MINIMAL
   del Pezzo surface of degree 6, not to a Severi-Brauer surface.


Two things survive, and it is worth being precise about which.

* **Step one still works, over $\mathbb{Q}$.** The triple $T_2$ is a Galois orbit defined over
  $F = \mathbb{Q}(\sqrt[3]{-2})$, and projection from its line is
  $$\pi(\mathbf{x}) = \big(x_0 - \theta x_1 \,:\, x_2 - \theta x_3\big) \in \mathbb{P}^1(F),
  \qquad \theta^3 = -2,$$
  giving $S \dashrightarrow R_{F/\mathbb{Q}}\mathbb{P}^1_F$, birational onto a degree-$6$ del Pezzo
  surface. Contracting the three lines costs nothing here, exactly as before.
* **Step two has no input at all**, by the computation above.

And the failure is not an artefact: a rational $\Gamma \in |2H - \ell|$ would be a genus-$0$ curve of
**odd** degree $3$, hence would have a rational point by Springer's theorem, hence
$S(\mathbb{Q}) \neq \emptyset$ — contradicting $S(\mathbb{Q}_7) = \emptyset$. So no such $\Gamma$ exists,
for any of the $72$ classes.

## 3. Case (b): a stable sixer and no stable triple

Here the class $\ell$ *does* exist, and the obstruction is precisely $\delta(\ell) \in \operatorname{Br}(k)[3]$.
The construction is the natural one: **blow up a non-trivial Severi–Brauer surface $P$ at a single
closed point of degree $6$** in general position. The six exceptional curves are a Galois-stable
sixer; if the degree-$6$ point has no sub-scheme of degree $3$ defined over $k$ — i.e. its Galois
group has no invariant $3$-subset, as in §1 — there is no stable triple; and $S(k) = \emptyset$ by
Nishimura, since $S$ is birational to $P$.

Its line combinatorics is entirely determined by the Galois action on the six points, so we can
exhibit it on the **split** model, blowing up $\mathbb{P}^2$ at the degree-$6$ point cut out by an
irreducible sextic with Galois group $S_6$. That model is $\mathbb{Q}$-rational, of course; twisting
$\mathbb{P}^2$ into a non-trivial $P$ leaves the whole picture below unchanged except that
$\delta(\ell)$ becomes non-zero.

In [8]:
U.<T> = QQ[]
fp = T^6 - T - 1
print("sextic:", fp, "  irreducible:", fp.is_irreducible(),
      "  Galois group:", fp.galois_group().structure_description())
Fk.<th> = NumberField(fp)
R3.<x,y,z> = QQ[]; RF = R3.change_ring(Fk)
def forms(m): return [x^i*y^j*z^(m-i-j) for i in range(m+1) for j in range(m+1-i)]
ptF = (Fk(1), th, th^4)
def condrows(mons):
    vals = [Fk(RF(m)(*ptF)) for m in mons]
    return [[QQ(v.list()[i]) for v in vals] for i in range(6)]
print("through the degree-6 point:  lines",
      Matrix(QQ,condrows(forms(1))).right_kernel().dimension(),
      "  conics", Matrix(QQ,condrows(forms(2))).right_kernel().dimension(),
      "  cubics", Matrix(QQ,condrows(forms(3))).right_kernel().dimension(),
      "  (general position)")

sextic: T^6 - T - 1   irreducible: True   Galois group: S6
through the degree-6 point:  lines 0   conics 0   cubics 4   (general position)


In [9]:
ker3 = Matrix(QQ, condrows(forms(3))).right_kernel().basis()
fcub = [sum(c*m for c,m in zip(v, forms(3))) for v in ker3]
fcub = [g*lcm(c.denominator() for c in g.coefficients()) for g in fcub]
S4.<Y0,Y1,Y2,Y3> = QQ[]; Yv = [Y0,Y1,Y2,Y3]
from itertools import combinations_with_replacement as cwr
idx = list(cwr(range(4),3))
rows = [[(fcub[i]*fcub[j]*fcub[k]).monomial_coefficient(m) for m in forms(9)] for i,j,k in idx]
rel = Matrix(QQ, rows).left_kernel().basis()
Fb = sum(c*prod(Yv[i] for i in tt) for c,tt in zip(rel[0], idx))
Fb = Fb*lcm(c.denominator() for c in Fb.coefficients())
print("S_b :", Fb, "= 0")
Ib = S4.ideal([Fb] + [Fb.derivative(v) for v in Yv])
print("smooth ?", Ib.dimension() - 1 == -1)

S_b : -Y0*Y1^2 + Y0^2*Y2 - 2*Y0*Y2^2 + Y2^3 + 2*Y0*Y1*Y3 + Y2^2*Y3 - Y0*Y3^2 - Y1*Y3^2 = 0
smooth ? True


In [10]:
# line structure: Galois acts through S_6 on the six exceptional curves
gensS6 = [induced(tuple(g(i+1)-1 for i in range(6))) for g in SymmetricGroup(6).gens()]
stb = lambda c: all(set(p[i] for i in c) == set(c) for p in gensS6)
print("Galois image on the six points is all of S_6, so on the 27 lines:")
print("   stable triples of skew lines:", sum(1 for c in triA if stb(c)))
print("   stable sixers               :", sum(1 for c in sixA if stb(c)))
print("   invariant blow-down classes :",
      len([D for D in BD if all(vector(ZZ,D) == vector(ZZ,tuple(
          Matrix(ZZ,[list(lc)] + [list(Ee[g(i+1)-1]) for i in range(6)]).transpose()*vector(ZZ,D)))
          for g in SymmetricGroup(6).gens())]))

Galois image on the six points is all of S_6, so on the 27 lines:
   stable triples of skew lines: 0
   stable sixers               : 2
   invariant blow-down classes : 2


Two invariant blow-down classes: $\ell$ (contract the six exceptional curves) and its partner
$2\ell - \sum E_i$ — precisely the pair whose Brauer classes are opposite, as in §II.7 of the
companion worksheet. On the split model both are represented by rational divisors and the recipe runs;
on the twisted model $\delta(\ell) = [P] \neq 0$, and:

* no rational $\Gamma \in |2H-\ell|$ exists — again because such a $\Gamma$ is a genus-$0$ curve of odd
  degree and would force $S(k) \neq \emptyset$;
* the net of quadrics through $\Gamma$ exists over the splitting field and nowhere smaller;
* and $S$ is birational to $P$, so it is not $k$-rational — there is no map to write down.

### Diagonal surfaces cannot do case (b) with no points

Worth recording, since diagonal surfaces are the obvious place to look: among diagonal cubic surfaces
a stable sixer forces **every** ratio $-a_j/a_i$ to be a cube, i.e. the surface is the Fermat cubic
surface up to scaling — which has rational points. So the case-(b) pointless examples really do have
to come from a twisted Severi–Brauer surface.

In [11]:
vals = [s*2^e*7^f for s in (1,-1) for e in range(3) for f in range(3)]
with_six, with_tri = [], 0
for a1 in vals:
    for a2 in vals:
        for a3 in vals:
            aa = (1,a1,a2,a3)
            _, pp, ss = lines_and_galois(aa)
            stq = lambda c: all(set(p[i] for i in c) == set(c) for p in pp)
            s6 = [c for c in cliq(ss,6) if stq(c)]
            if s6: with_six.append(aa)
            elif any(stq(c) for c in cliq(ss,3)): with_tri += 1
print("diagonal surfaces with coefficients in +-2^i 7^j (i,j < 3):")
print("   with a stable sixer      :", len(with_six), "->", sorted(set(with_six))[:4], "...")
print("   all have |a_i| = 1 ?     ", all(all(abs(c)==1 for c in aa) for aa in with_six))
print("   with a stable triple only:", with_tri)
print()
print("pointless at 7 among those with a stable sixer:",
      len([aa for aa in with_six if pointless_at(aa,7)]))

diagonal surfaces with coefficients in +-2^i 7^j (i,j < 3):
   with a stable sixer      : 8 -> [(1, -1, -1, -1), (1, -1, -1, 1), (1, -1, 1, -1), (1, -1, 1, 1)] ...
   all have |a_i| = 1 ?      True
   with a stable triple only: 1728

pointless at 7 among those with a stable sixer: 0


## Summary

| | case (a) | case (b) |
|---|---|---|
| stable set of skew lines | three, no sixer | a sixer, no triple |
| example | $x^3+2y^3+7z^3+14w^3$ over $\mathbb{Q}$ | blow-up of a non-trivial Severi–Brauer surface at a degree-$6$ point |
| rational points | none ($S(\mathbb{Q}_7) = \emptyset$) | none (Nishimura) |
| invariant blow-down class | **none** ($0$ of $72$) | two, $\ell$ and $2\ell-\sum E_i$ |
| $\operatorname{rank}\operatorname{Pic}(\bar S)^G$ | $2$ | $2$ |
| birational to | a **minimal** del Pezzo surface of degree $6$ | the Severi–Brauer surface $P$ |
| where the recipe dies | in $\operatorname{Pic}(\bar S)^G$: no $\ell$ to start from | in $\operatorname{Br}(k)[3]$: $\delta(\ell) = [P] \neq 0$ |
| step one (contract to a $dP_6$) | still works, over $\mathbb{Q}$ | not available (no stable triple) |

Both failures have the same clean witness: a rational $\Gamma \in |2H-\ell|$ is a genus-$0$ curve of
**odd degree $3$**, so by Springer's theorem it would carry a rational point and force
$S(k) \neq \emptyset$. So in both cases the absence of a rational $\Gamma$ is not a limitation of the
linear algebra — it is the statement that $S$ is not $k$-rational.

The combinatorial point is settled in §1: both configurations occur. A stable triple with no stable
sixer happens as soon as Galois swaps the two triples of lines skew to the given three — realised
arithmetically by $x^3+2y^3+7z^3+14w^3$. A stable sixer with no stable triple happens as soon as the
Galois group acts on the six lines without an invariant $3$-subset, which $S_6$ and many of its
subgroups do.